In [1]:
"""
<다중회귀모델(Multiple Regression)>
 - 여러 개의 특성(항목, 컬럼, 퓨처, 피처 모두 같은 의미)을 사용한 회귀모델
 - 특성이 많을 수록 모델은 집중력과 복잡도가 높아짐
 - 단점으로는, 훈련 시간이 오래 걸릴 수 있으며,
              시스템 성능에 따라 처리 시간이 결정됨
 - 다중회귀모델 산술식(y) = aX1 + bX2 + cX3 .... nXn + y절편
 - a, b, c ...n : 기울기(또는 계수 or 가중치라고도 칭함)
"""

'\n<다중회귀모델(Multiple Regression)>\n - 여러 개의 특성(항목, 컬럼, 퓨처, 피처 모두 같은 의미)을 사용한 회귀모델\n - 특성이 많을 수록 모델은 집중력과 복잡도가 높아짐\n - 단점으로는, 훈련 시간이 오래 걸릴 수 있으며,\n              시스템 성능에 따라 처리 시간이 결정됨\n - 다중회귀모델 산술식(y) = aX1 + bX2 + cX3 .... nXn + y절편\n - a, b, c ...n : 기울기(또는 계수 or 가중치라고도 칭함)\n'

### 라이브러리 불러들이기

In [2]:
# 숫자 처리
import numpy as np

# 데이터 처리
import pandas as pd

# 훈련:검증:테스트 데이터 분리 처리
from sklearn.model_selection import train_test_split

# 훈련모델
from sklearn.linear_model import LinearRegression

# 특성 늘리는 클래스
from sklearn.preprocessing import PolynomialFeatures

# 표준화 데이터 스케일링
from sklearn.preprocessing import StandardScaler

In [3]:
"""
<분석 과정>
데이터 수집 > 전처리 > 현황분석(EDA) 시각화 > [후처리 > 모델링] > 모델 선정 및 파일 저장
"""

'\n<분석 과정>\n데이터 수집 > 전처리 > 현황분석(EDA) 시각화 > [후처리 > 모델링] > 모델 선정 및 파일 저장\n'

### 데이터셋 파일 읽어들이기

In [4]:
### 사용할 데이터셋 파일명 : 03_농어의_길이_높이_두께_데이터.csv
# - 읽어들인 데이터는 df 변수에 저장
# - 데이터 정보 확인하기

df = pd.read_csv("./dataset/03_농어의_길이_높이_두께_데이터.csv")
df
df.info()
df.describe()
df.head(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   length  56 non-null     float64
 1   height  56 non-null     float64
 2   width   56 non-null     float64
dtypes: float64(3)
memory usage: 1.4 KB


,length,height,width
0,8.4,2.11,1.41
1,13.7,3.53,2.00
2,15.0,3.82,2.43


In [5]:
### 농어의 길이
perch_length= np.array(
    [8.4, 13.7, 15.0, 16.2, 17.4, 18.0, 18.7, 19.0, 19.6, 20.0,
     21.0, 21.0, 21.0, 21.3, 22.0, 22.0, 22.0, 22.0, 22.0, 22.5,
     22.5, 22.7, 23.0, 23.5, 24.0, 24.0, 24.6, 25.0, 25.6, 26.5,
     27.3, 27.5, 27.5, 27.5, 28.0, 28.7, 30.0, 32.8, 34.5, 35.0,
     36.5, 36.0, 37.0, 37.0, 39.0, 39.0, 39.0, 40.0, 40.0, 40.0,
     40.0, 42.0, 43.0, 43.0, 43.5, 44.0]
     )


### 농어 무게
perch_weight = np.array(
    [5.9, 32.0, 40.0, 51.5, 70.0, 100.0, 78.0, 80.0, 85.0, 85.0,
     110.0, 115.0, 125.0, 130.0, 120.0, 120.0, 130.0, 135.0, 110.0,
     130.0, 150.0, 145.0, 150.0, 170.0, 225.0, 145.0, 188.0, 180.0,
     197.0, 218.0, 300.0, 260.0, 265.0, 250.0, 250.0, 300.0, 320.0,
     514.0, 556.0, 840.0, 685.0, 700.0, 700.0, 690.0, 900.0, 650.0,
     820.0, 850.0, 900.0, 1015.0, 820.0, 1100.0, 1000.0, 1100.0,
     1000.0, 1000.0]
     )

# np.array : 그냥 리스트라 보면 됨.

In [6]:
"""
- 전처리는 할 게 없다는 가정하에 진행
<분석 주제 정하기>
- 농어의 길이, 두께, 너비를 이용해서 무게 예측하기

<데이터 구분>
- 독립변수(X) : 길이, 두께, 너비
- 종속변수(y) : 무게
"""

print(df.shape, perch_weight.shape)
print(type(df), type(perch_weight))

(56, 3) (56,)
<class 'pandas.core.frame.DataFrame'> <class 'numpy.ndarray'>


### 독립변수(데이터프레임 타입)를 종속변수의 넘파이(array=리스트) 타입으로 변환하기

In [7]:
### 데이터 프레임 타입을 넘파이 배열(리스트)타입으로 변환
# - 데이터프레임 내에 넘파이 배열(리스트)로 변환하는 함수(to numpy())

perch_full = df.to_numpy()
print(perch_full.shape, perch_weight.shape)
perch_full
print(type(perch_full))

(56, 3) (56,)
<class 'numpy.ndarray'>


In [8]:
# 훈련 : 테스트 / 7 : 3으로 나누기
 
train_input, test_input, train_target, test_target = train_test_split(perch_full, perch_weight, test_size=0.3, random_state=42)

print(train_input.shape, train_target.shape)
print(test_input.shape, test_target.shape)


(39, 3) (39,)
(17, 3) (17,)


### 훈련(학습)하기

In [ ]:
### 모델 클래스 생성부터 ~ 오차 확인까지 해주세요.

# 모델 클래스 생성
lr = LinearRegression()
lr
# 훈련시키기
lr.fit(train_input, train_target)

# 정확도 확인(훈련 정확도 & 테스트 정확도)
train_score = lr.score(train_input, train_target)
test_score = lr.score(test_input, test_target)
gap = train_score - test_score
print(train_score, test_score, gap)

# 예측하기(테스트 예측)
pred = lr.predict(test_input)
print(pred)

# 오차확인(테스트 오차)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
mae = mean_absolute_error(test_target, pred)
mse = mean_squared_error(test_target, pred)
r2 = r2_score(test_target, pred)

print(mae, mse, r2)

0.9537065271284176 0.886342083634778 0.06736444349363968
[-328.54944233   48.99648009  328.530777    168.01071374  147.28466932
  790.89418753  383.33837613  247.89949679  814.06695812  123.2000108
  981.8543067   -37.63667416  356.2229673   419.56345249   48.15564696
  140.06819759   46.70641917]
68.56651514640546
68.56651514640546 10259.932540156356 0.886342083634778


## 특성(컬럼, 퓨처) 늘리기
- 퓨처 엔지니어링 중 하나

In [10]:
"""
## 다중회귀모델은 많은 특성을 이용할 수 있는 모델로,
 - 좋은 성능이 나타났음에도 좀 더 특성을 늘려서 성능향상이 가능한지 확인


<특성을 생성하는 라이브러리>
  - 패키기 : sklearn.preprocessing
  - 사용모델(클래스) : PolynomialFeatures(), 실제 모델은 아님
                    : "변환기" 모델이라고 칭합니다.
  - 사용함수 : fit(), transform(), fit_transform()
            -> 훈련데이터와 검증데이터, 테스트데이터를 모두 변환할 경우에는 아래 순서에 따름
              1. 훈련데이터로 fit(훈련독립변수) 수행
                - 변환할 패턴 찾기

              2. transform(훈련 or 검증 or 테스트 독립변수) 함수로 훈련/검증/테스트 독립변수를
                 각각 수행(검증 및 테스트에 대해서는 fit을 수행하지 않음)
                - 찾은 변환 패턴으로 데이터 변환하기

            -> 훈련데이터만을 이용하여 간단히 모델성능을 확인하고자 할 때는
               빠른 수행을 위해 fit_transform()으로 1번과 2번을 동시에 수행하기도 함

            * 훈련 데이터에서는 스케일링을 작업하고 검증 및 테스트에서는 원본 데이터를 쓰면 의미가 없음
            * 훈련 데이터의 유형은 3개이나 검증 및 테스트에서는 데이터 유형을 5개로 하면 오류남
 
<특성 생성 순서>
 - 변환기 모델 생성 : PolynomialFeatures()
 - 변환 패턴 찾기 : fit(훈련 독립변수)
 - 찾은 패턴으로 변환하기 : transform(훈련/검증/테스트 독립변수)


<특성을 추가하는 데이터>
 - 훈련/테스트, 또는 훈련/검증/테스트 데이터에서 독립변수만 변환합니다.
 - 종속변수는 원본 유지 합니다.
 ** 종속변수는 정답! 변환시키면 안 됨.
"""


'\n## 다중회귀모델은 많은 특성을 이용할 수 있는 모델로,\n - 좋은 성능이 나타났음에도 좀 더 특성을 늘려서 성능향상이 가능한지 확인\n\n\n<특성을 생성하는 라이브러리>\n  - 패키기 : sklearn.preprocessing\n  - 사용모델(클래스) : PolynomialFeatures(), 실제 모델은 아님\n                    : "변환기" 모델이라고 칭합니다.\n  - 사용함수 : fit(), transform(), fit_transform()\n            -> 훈련데이터와 검증데이터, 테스트데이터를 모두 변환할 경우에는 아래 순서에 따름\n              1. 훈련데이터로 fit(훈련독립변수) 수행\n                - 변환할 패턴 찾기\n\n              2. transform(훈련 or 검증 or 테스트 독립변수) 함수로 훈련/검증/테스트 독립변수를\n                 각각 수행(검증 및 테스트에 대해서는 fit을 수행하지 않음)\n                - 찾은 변환 패턴으로 데이터 변환하기\n\n            -> 훈련데이터만을 이용하여 간단히 모델성능을 확인하고자 할 때는\n               빠른 수행을 위해 fit_transform()으로 1번과 2번을 동시에 수행하기도 함\n\n            * 훈련 데이터에서는 스케일링을 작업하고 검증 및 테스트에서는 원본 데이터를 쓰면 의미가 없음\n            * 훈련 데이터의 유형은 3개이나 검증 및 테스트에서는 데이터 유형을 5개로 하면 오류남\n \n<특성 생성 순서>\n - 변환기 모델 생성 : PolynomialFeatures()\n - 변환 패턴 찾기 : fit(훈련 독립변수)\n - 찾은 패턴으로 변환하기 : transform(훈련/검증/테스트 독립변수)\n\n\n<특성을 추가하는 데이터>\n - 훈련/테스트, 또는 훈련/검증/테스트 데이터에서 독립변수만 변환합니다.

### 특성 증가 예시

In [11]:
### 샘플 데이터(독립변수)
temp_data = [[2, 3, 4]]

### 1. 변환기 클래스(특성 생성하는 클래스) 생성하기
# - include_bias=False : bias는 절편을 의미하며 0으로 보고 진행함(절편 생략 = False 지정)
poly = PolynomialFeatures(include_bias=False)
poly

### 2. 패턴 찾기
# - 모델 훈련시에 종속변수도 함께 넣어줬으나, 변환기 클래스에는 종속변수 넣지 않음.
# - 독립변수의 데이터 변환이 목적.
poly.fit(temp_data)


### 3. 찾은 패턴을 이용하여 -> 독립변수의 특성 변환 시키기(특성 생성)
temp_poly = poly.transform(temp_data)
print(temp_poly)

### fit()이 찾아낸 패턴(공식) 추출하기
poly.get_feature_names_out()

[[ 2.  3.  4.  4.  6.  8.  9. 12. 16.]]


array(['x0', 'x1', 'x2', 'x0^2', 'x0 x1', 'x0 x2', 'x1^2', 'x1 x2',
       'x2^2'], dtype=object)

In [12]:
### 샘플 데이터(독립변수)
temp_data = [[2, 3, 4]]

### 1. 변환기 클래스(특성 생성하는 클래스) 생성하기
# - include_bias=False : bias는 절편을 의미하며 0으로 보고 진행함(절편 생략 = False 지정)
# - degree : 특성 증가할 차원 정의(보통 최대 4제곱승 까지 사용)
poly = PolynomialFeatures(degree=4, include_bias=False)
poly

### 2. 패턴 찾기
# - 모델 훈련시에 종속변수도 함께 넣어줬으나, 변환기 클래스에는 종속변수 넣지 않음.
# - 독립변수의 데이터 변환이 목적.
poly.fit(temp_data)


### 3. 찾은 패턴을 이용하여 -> 독립변수의 특성 변환 시키기(특성 생성)
temp_poly = poly.transform(temp_data)
print(temp_poly)

### fit()이 찾아낸 패턴(공식) 추출하기 / 가끔 입사 문제 냄. 패턴 찾아내기
poly.get_feature_names_out()

[[  2.   3.   4.   4.   6.   8.   9.  12.  16.   8.  12.  16.  18.  24.
   32.  27.  36.  48.  64.  16.  24.  32.  36.  48.  64.  54.  72.  96.
  128.  81. 108. 144. 192. 256.]]


array(['x0', 'x1', 'x2', 'x0^2', 'x0 x1', 'x0 x2', 'x1^2', 'x1 x2',
       'x2^2', 'x0^3', 'x0^2 x1', 'x0^2 x2', 'x0 x1^2', 'x0 x1 x2',
       'x0 x2^2', 'x1^3', 'x1^2 x2', 'x1 x2^2', 'x2^3', 'x0^4', 'x0^3 x1',
       'x0^3 x2', 'x0^2 x1^2', 'x0^2 x1 x2', 'x0^2 x2^2', 'x0 x1^3',
       'x0 x1^2 x2', 'x0 x1 x2^2', 'x0 x2^3', 'x1^4', 'x1^3 x2',
       'x1^2 x2^2', 'x1 x2^3', 'x2^4'], dtype=object)

### 실제 사용하는 훈련 및 테스트 데이터를 이용하여 특성 생성하기

In [13]:
### 특성 생성할 차원은 2(degree=2)로
# - 실제 변수 : train_input, test_input
# - 생성 변수명 : train_poly, test_poly
train_input.shape, test_input.shape

# 클래스 생성
poly = PolynomialFeatures(degree=2, include_bias=False) # 클래스 생성시 차원과 절편 설정!
poly

# 훈련 독립변수로 패턴 찾기! 훈련 독립변수만
poly.fit(train_input)

# 훈련 및 테스트 독립변수 변환하기
train_poly = poly.transform(train_input)
test_poly = poly.transform(test_input)

print(train_input.shape, test_input.shape)
print(train_poly.shape, test_poly.shape)
poly.get_feature_names_out()

(39, 3) (17, 3)
(39, 9) (17, 9)


array(['x0', 'x1', 'x2', 'x0^2', 'x0 x1', 'x0 x2', 'x1^2', 'x1 x2',
       'x2^2'], dtype=object)

### 모델 훈련 시키기

In [14]:
### 선형회귀(다중회귀모델) 훈련 시키기

# - 훈련 후 훈련 및 테스트 정확도(결정계수) 확인



# - 과적합 여부 확인
"""
과적합 아님
"""

# - 테스트 데이터로 예측하기


# - MAE, MSE 오차 확인하기




###<모델 훈련 순서> `필히 외울 것!`
# 1. 모델 클래스 생성
lr = LinearRegression()
# 2. 훈련 독립변수와 종속변수로 훈련시키기(fit)
lr.fit(train_poly, train_target)
# 3. 훈련 및 테스트 데이터로 훈련결과 확인 : score()
train_score = lr.score(train_poly, train_target)
test_score = lr.score(test_poly, test_target)
gap = train_score - test_score
print(train_score, test_score, gap)
#4. 과적합 여부 확인: 확인 > 테스트
if (train_score > test_score) & (gap <= 0.1):
    print("과적합 아님")
elif (train_score > test_score) & (gap > 0.1):
    print("train_score > test_score 이나 차이가 0.1 초과로 과대적합")
elif (train_score < test_score):
    print("과소적합")

#5. 예측하기 : predict()
pred_test = lr.predict(test_poly)
pred_test

#6. 예측결과를 이용하여 오차 확인: MAE,
# 평균"절대"오차
mae = mean_absolute_error(test_target, pred_test)
# 평균제곱오차
mse = mean_squared_error(test_target, pred_test)

print(mae, mse)

# 순수 데이터 모델: 0.9537065271284176 0.886342083634778 0.06736444349363968
# 2차원 변형 데이터 모델 : 0.9898271546307027 0.9713771600629659 0.01844999456773677

### (해석)
# - 특성을 증가하기 전과 후를 비교했을 때
# - 특성을 증가한 경우 훈련 및 테스트 정확도가 높으며,
# - 과적합에 대해서도 나타나지 않으며, 오차도 매우 낮아졌음
# - 특성 증가후 차원2일때와 3일때를 비교한 결과
# - 3일때의 훈련 정확도가 매우 높게 나타났으나,
# - 과적합의 경우 차원 2일때 매우 낮으며, 오차도 낮게 나타났음
# - 따라서, 농어의 길이, 두께, 너비를 이용하여 무게를 예측하는 모델로는
# - 특성의 차원2로 증가한 다중회귀모델이 가장 일반화되어,
#   매우 훌륭한 모델로 판단됨(현재까지..)


0.9898271546307027 0.9713771600629659 0.01844999456773677
과적합 아님
30.2168895903429 2583.791928034344


In [15]:
# <성능 높이는 방법> / `필히 외울 것!`
# - 데이터양 늘리기(행, 열 둘 다 해당함.) / 실제 데이터를 더 수집하는 것은 so hard -> 최후 수단
# - 하이퍼파라메터 튜닝(선형회귀에서는 튜닝 대상 모델이 아님, 변수 없음)
# - 다른모델 사용
# - 복잡도(x값=특성=컬럼=퓨처) 늘리기
# - 정규화(표준화)하기


In [16]:
# 농어의 길이, 두께, 너비를 이용하여 무게를 예측
# - 데이터의 단위가 일의자리~천의자리까지 편차가 큼
# - 따라서 데이터 스케일링 작업을 진행해 봄(성능에 영향을 미치는지 확인)
train_poly

array([[  17.4   ,    4.59  ,    2.94  ,  302.76  ,   79.866 ,   51.156 ,
          21.0681,   13.4946,    8.6436],
       [  36.    ,   10.61  ,    6.74  , 1296.    ,  381.96  ,  242.64  ,
         112.5721,   71.5114,   45.4276],
       [  25.    ,    6.44  ,    3.68  ,  625.    ,  161.    ,   92.    ,
          41.4736,   23.6992,   13.5424],
       [  40.    ,   11.93  ,    7.11  , 1600.    ,  477.2   ,  284.4   ,
         142.3249,   84.8223,   50.5521],
       [  39.    ,   12.43  ,    7.35  , 1521.    ,  484.77  ,  286.65  ,
         154.5049,   91.3605,   54.0225],
       [  43.    ,   11.93  ,    7.28  , 1849.    ,  512.99  ,  313.04  ,
         142.3249,   86.8504,   52.9984],
       [  22.    ,    5.64  ,    3.52  ,  484.    ,  124.08  ,   77.44  ,
          31.8096,   19.8528,   12.3904],
       [  20.    ,    5.08  ,    2.77  ,  400.    ,  101.6   ,   55.4   ,
          25.8064,   14.0716,    7.6729],
       [  22.    ,    6.11  ,    3.52  ,  484.    ,  134.42  ,   77.44  

### 데이터 스케일링 처리하기

In [17]:
# 표준화(StandardScaler)만 수행해보기
# - 훈련 및 테스트 데이터 모두 스케일링 후
# - 다중회귀모델로 재훈련 시켜서 성능 확인하기

# <순서>
# 클래스 생성
scaler = StandardScaler()
# 훈련 독립변수 패턴 찾기
train_std = scaler.fit(train_poly)
# 훈련 및 테스트 독립변수 모두 찾은 패턴으로 변환하기
train_std = scaler.transform(train_poly)
test_std = scaler.transform(test_poly) ## 표준화에서도 훈련 데이터에만 fit 함

###<모델 훈련 순서> `필히 외울 것!`
# 1. 모델 클래스 생성
lr = LinearRegression()
# 2. 훈련 독립변수와 종속변수로 훈련시키기(fit)
lr.fit(train_std, train_target)
# 3. 훈련 및 테스트 데이터로 훈련결과 확인 : score()
train_score = lr.score(train_std, train_target)
test_score = lr.score(test_std, test_target)
gap = train_score - test_score
print(train_score, test_score, gap)
#4. 과적합 여부 확인: 확인 > 테스트
if (train_score > test_score) & (gap <= 0.1):
    print("과적합 아님")
elif (train_score > test_score) & (gap > 0.1):
    print("train_score > test_score 이나 차이가 0.1 초과로 과대적합")
elif (train_score < test_score):
    print("과소적합")

#5. 예측하기 : predict()
pred_test = lr.predict(test_std)
pred_test

#6. 예측결과를 이용하여 오차 확인: MAE,
# 평균"절대"오차
mae = mean_absolute_error(test_target, pred_test)
# 평균제곱오차
mse = mean_squared_error(test_target, pred_test)

print(mae, mse)

### (해석)
# - 기존 가장 좋은 시점(차원2)의 정확도, 과적합여부, 오차와 비교 했을 때
# - 매우 미세한 차이로 정규화를 하지 않았을 때의 모델 성능이 더 좋았기에
# - 차원 2일때의 모델로 선정하는 것이 타당함

0.9898271546307026 0.9713771600629622 0.018449994567740435
과적합 아님
30.216889590344287 2583.791928034682


In [18]:
# <성능 높이는 방법>
# - 데이터양 늘리기


# - 하이퍼파라메터 튜닝(선형회귀에서는 튜닝 대상 모델이 아님, 변수 없음)
# - 다른모델 사용 -> 튜닝이 가능한 개선된 모델 사용해보기...
#   * <하이퍼파라메터 튜닝 가능 모델>
#     - KNN(현장에서는 안 씀. 이웃에 대한 한계), **릿지(Ridge), **라소(Lasso), 앙상블모델 전체
#     - 훈련의 성능(과적합)에 영향을 미치는 변수의 값을 이용하여 조정하는 모델들임


# - 복잡도(x값=특성=컬럼=퓨처) 늘리기 - ok
# - 정규화(표준화)하기 - ok

## 하이퍼파라메터가 적용된 다른 모델 사용하기(릿지, 라쏘)

### 라이브러리 불러들이기

In [19]:
from sklearn.linear_model import Ridge, Lasso
## 회귀에서만 사용 가능

In [20]:
"""
<사용 모델>
 - 사용 패키지 : sklearn.linear_model
 - 사용 모델(클래스) : Ridge, Lasso
 - 훈련에 사용되는 함수 : 기존 모델 훈련과 동일
 - 사용되는 하이퍼파라메터 : 규제강도: c (복잡도, 집중력)라는 개념이 적용됨


<모델 훈련시 주의할점>
 - 하이퍼파라메터의 값은 사람이 조정할 수 있지만,
 -> 사람은 모델 훈련에는 개입하지 않습니다.(fit 개입 불가)
"""

'\n<사용 모델>\n - 사용 패키지 : sklearn.linear_model\n - 사용 모델(클래스) : Ridge, Lasso\n - 훈련에 사용되는 함수 : 기존 모델 훈련과 동일\n - 사용되는 하이퍼파라메터 : 규제강도: c (복잡도, 집중력)라는 개념이 적용됨\n\n\n<모델 훈련시 주의할점>\n - 하이퍼파라메터의 값은 사람이 조정할 수 있지만,\n -> 사람은 모델 훈련에는 개입하지 않습니다.(fit 개입 불가)\n'

### 릿지(Ridge) 모델로 훈련하기

In [ ]:
# 1. 훈련모델 클래스 생성
# 2. 생성된 클래스 모델로 훈련시키기(fit)
# 3. 정확도(결정계수) 확인
# 4. 과적합 확인
# 5. 예측
# 6. 오차 확인(MAE만)

#<순서>

# 클래스 생성
rg = Ridge()

# 훈련시키기
rg.fit(train_std, train_target)

# 정확도 확인
train_score = rg.score(train_std, train_target)
test_score = rg.score(test_std, test_target)
gap = train_score - test_score
print(train_score, test_score, gap)

# 과적합 확인
if (train_score > test_score) & (gap <= 0.1):
    print("과적합 아님")
elif (train_score > test_score) & (gap > 0.1):
    print("과대적합")
elif (train_score < test_score):
    print("과소적합")

# 예측(훈련 및 테스트)
train_pred = rg.predict(train_std) 
test_pred = rg.predict(test_std)

#오차 확인
train_mae = mean_absolute_error(train_target, train_pred)
test_mae = mean_absolute_error(test_target, test_pred)
print(f"훈련 평균절대오차 = {train_mae}, 테스트 평균절대오차 = {test_mae}")

### (해석)
# - 기존 가장 좋은 모델인 차원2의 성능과 비교 했을 때
# - 훈련정확도는 미세하게 낮아졌으나,
# - 테스트정확도는 더욱 좋와 졌음.(일반화가 매우 잘 되었음.)
# - 즉, 과적합이 최소화 되었음.
# - 오차 또한 낮아졌음.
# - 따라서, 가장 일반화가 잘된 모델은 릿지(Ridge)모델로 판단됨.
# - 다만, 선형회귀모델의 차원2일 때의 모델과 릿지모델과 선별은
#         두 모델의 성능차이는 매우 낮기 때문에 분석가의 판단에 따르면 됨.
# - `타당성`은 `일반화`를 우선시하고 있음.


0.9849041294689239 0.984517359161522 0.00038677030740186247
과적합 아님
훈련 평균절대오차 = 30.787165623175987, 테스트 평균절대오차 = 29.25391147401801


### 라소(Lasso) 모델도 훈련시키기

In [24]:
# <모델링 순서>
# 1. 클래스 생성
la = Lasso()
# 2. 훈련시키기
la.fit(train_std, train_target)
# 3. 정확도 측정
train_score = la.score(train_std, train_target)
test_score = la.score(test_std, test_target)
gap = train_score - test_score
print(train_score, test_score, gap)
# 4.과적합 여부
if (train_score > test_score) & (gap <= 0.1):
    print("과적합 아님")
elif (train_score > test_score) & (gap > 0.1):
    print("과대적합")
elif train_score < test_score:
    print("과소적합")
# 5. 예측해보기(훈련 및 테스트 데이터로)
train_pred = la.predict(train_std)
test_pred = la.predict(test_std)
# 6. 오차 검증(평균절대오차)
train_mae = mean_absolute_error(train_target, train_pred)
test_mae = mean_absolute_error(test_target, test_pred)

print(train_mae, test_mae)



0.9861305259897015 0.9863220281055399 -0.00019150211583840449
과소적합
29.633314460397507 26.87838445017345


### 하이퍼파라메터 튜닝하기(규제강도 값 조정)

In [ ]:
"""
<릿지, 라쏘에 사용되는 하이퍼파라메터>
 - 규제 강도 : 훈련에 집중을 강하게 또는 느슨하게 시키기 위한 값
 - 규제 속성명(변수명) : alpha
 - 규제 값의 범위 : 0.001 ~ 100 상이의 값중에 1개 사용
 - 규제 기본값(디폴트값, 생략가능) : 1
 - 보통 사용되는 값 : 0.001, 0.01, 0.1, 1, 10, 100 (기준은 1을 기준으로 비교)

<규제 값의 성격>
 - alpha 값이 작을 수록 : 훈련 정확도는 낮아지고, 과적합에 도움을 줄 수 있음
 - alpha 값이 높을 수록 : 훈련 정확도는 높아지며, 과적합에 도움이 안될 가능성이 높음
"""

Ridge()

,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


### 릿지(Ridge) 모델에 규제강도 10 적용해보기

In [ ]:
# 1. 훈련모델 클래스 생성
# 2. 생성된 클래스 모델로 훈련시키기(fit)
# 3. 정확도(결정계수) 확인
# 4. 과적합 확인
# 5. 예측
# 6. 오차 확인(MAE만)

#<순서>

# 클래스 생성 : 보통 클래스 생성시 하이퍼파라메터도 함께 정의
rg = Ridge(alpha=10)
# 또는 클래스 생성 후에 정의도 가능
# -> rg.alpha = 10

# 훈련시키기
rg.fit(train_std, train_target)

# 정확도 확인
train_score = rg.score(train_std, train_target)
test_score = rg.score(test_std, test_target)
gap = train_score - test_score
print(train_score, test_score, gap)

# 과적합 확인
if (train_score > test_score) & (gap <= 0.1):
    print("과적합 아님")
elif (train_score > test_score) & (gap > 0.1):
    print("과대적합")
elif (train_score < test_score):
    print("과소적합")

# 예측(훈련 및 테스트)
train_pred = rg.predict(train_std) 
test_pred = rg.predict(test_std)

#오차 확인
train_mae = mean_absolute_error(train_target, train_pred)
test_mae = mean_absolute_error(test_target, test_pred)
print(f"훈련 평균절대오차 = {train_mae}, 테스트 평균절대오차 = {test_mae}")
